In [1]:
!pip install  langchain_community langchain
!pip install tabulate pandas

In [2]:
import re
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.geeksforgeeks.org/devops/amazon-aurora/")
contents = loader.load()

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
# Create the splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
   # separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(contents)

print(f"Created {len(chunks)} chunks")
#for i, chunk in enumerate(chunks, 1):
#    chunk.page_content



text = "\n".join([doc.page_content for doc in contents])
cleaned = re.sub(r'\n+', '\n', text)

cleanchunks = splitter.split_text(cleaned)
print(f"Created  after cleaned {len(cleanchunks)} chunks")
# Step 4: Wrap chunks into Document objects
documents = [Document(page_content=chunk) for chunk in cleanchunks]



Created 26 chunks
Created  after cleaned 26 chunks


In [4]:
from langchain_openai import OpenAIEmbeddings 

# Initialize embeddings
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# Generate embedding for a query
vector = embedding.embed_query("What is Aurora?")

print(len(vector))       # length of the embedding vector
print(vector[:10])       # preview first 10 values


1536
[-0.02918471951042551, -0.05055428400550695, -0.021827483734261757, 0.01536319047449962, -0.04881419089662174, -0.01308122626591771, 0.010661886285581705, -0.035717700656064694, 0.02019423844785196, 0.00748316356693834]


In [5]:
from langchain_community.vectorstores import FAISS
from langchain.vectorstores import Chroma


db = FAISS.from_documents(contents, embedding)

# Save FAISS index only (no pickle docstore)
db.save_local("my_vectorstore", index_name="faiss_index")




# Later, reload index and reattach your own docstore
db = FAISS.load_local(
    "my_vectorstore",
    embedding,
    index_name="faiss_index",
    allow_dangerous_deserialization=True  # stays safe// 
)

In [6]:
import re
q ="Amazon Aurora"
results  = db.similarity_search(q)
for doc in results:
    print("_______________________________________")
    collapsed = re.sub(r'\n+', '\n', doc.page_content)
    print(collapsed)
    print("_______________________________________")
    
results  = db.similarity_search(q, k=3)
for doc in results:
    print("_______________________________________")
    collapsed = re.sub(r'\n+', '\n', doc.page_content)
    print(collapsed)
    print("_______________________________________")

results = db.similarity_search_with_score(q, k=5)
print(results);
for doc, score in results:
    print("_______________________________________")
    print(f"Score: {score}")
    collapsed = re.sub(r'\n+', '\n', doc.page_content)
    print(collapsed)
    print("_______________________________________")

results = db.similarity_search_with_relevance_scores(q, k=5)
for doc, score in results:
    print("_______________________________________")
    print(f"Score: {score}")
    collapsed = re.sub(r'\n+', '\n', doc.page_content)
    print(collapsed)
    print("_______________________________________")
    

_______________________________________
Amazon Aurora - GeeksforGeeksCoursesTutorialsPracticeJobsDevOpsCloud ComputingGitAWSDockerKubernetesMicrosoft AzureGoogle Cloud PlatformPythonGolangOperating SystemComputer NetworkShare Your ExperiencesDevOps BasicsWhat is DevOpsDevOps LifecycleThe Evolution of DevOps - 3 Major Trends for FutureVersion ControlVersion Control SystemsMerge Strategies in GitWhich Version Control System Should I Choose?CI & CDWhat is CI/CD?Understanding Deployment AutomationContainerizationWhat is Docker?What is Dockerfile Syntax?OrchestrationKubernetes - Introduction to Container OrchestrationFundamental Kubernetes Components and their role in Container OrchestrationHow to Use AWS ECS to Deploy and Manage Containerized Applications?Infrastructure as Code (IaC)Infrastructure as Code (IaC)Introduction to TerraformAWS CloudformationMonitoring and LoggingWorking with Prometheus and Grafana Using HelmWorking with Monitoring and Logging ServicesMicrosoft Teams vs SlackSec

In [7]:
vectordb = Chroma(persist_directory="db", embedding_function=embedding)
vectordb.add_documents(contents)
q =" Amazon Aurora"
results  = vectordb.similarity_search(q)
for doc in results:
    print("_______________________________________")
    collapsed = re.sub(r'\n+', '\n', doc.page_content)
    print(collapsed)
    print("_______________________________________")
    
results  = vectordb.similarity_search(q, k=3)
for doc in results:
    print("_______________________________________")
    collapsed = re.sub(r'\n+', '\n', doc.page_content)
    print(collapsed)
    print("_______________________________________")

results = vectordb.similarity_search_with_score(q, k=5)
print(results);
for doc, score in results:
    print("_______________________________________")
    print(f"Score: {score}")
    collapsed = re.sub(r'\n+', '\n', doc.page_content)
    print(collapsed)
    print("_______________________________________")

results = vectordb.similarity_search_with_relevance_scores(q, k=5)
for doc, score in results:
    print("_______________________________________")
    print(f"Score: {score}")
    collapsed = re.sub(r'\n+', '\n', doc.page_content)
    print(collapsed)
    print("_______________________________________")

_______________________________________

Relational Database – Amazon Aurora MySQL PostgreSQL – AWS
Skip to main content
     
Filter: All
 
English
Contact us
AWS Marketplace
Support  
My account  
     
Search
Filter: All
 
Sign in to console
Create account
Amazon Aurora
Overview
Engines
Features
Pricing
Resources
More
Databases
Amazon RDS
Amazon Aurora
Amazon Aurora
Unparalleled high performance and availability at global scale for PostgreSQL, MySQL, and DSQL
Get started with Aurora
Connect with an Aurora specialist
What is Aurora
Aurora has 5x the throughput of MySQL and 3x of PostgreSQL with full PostgreSQL and MySQL compatibility. Aurora also offers DSQL, the fastest distributed SQL database that is PostgreSQL-compatible. Aurora is designed for up to 99.999% multi-Region availability. With Aurora DSQL, Aurora provides virtually unlimited scale in and across regions with no infrastructure management. Aurora offers broad compliance standards and enterprise security capabilities, an

In [10]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
# Step 5: Build RetrievalQA pipeline
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(search_kwargs={"k": 3}),
    chain_type="stuff"
)

# Step 6: Query the system
query = "Explain Amazon Aurora architecture and its advantages."
answer = qa.invoke(query)


print("===== Query =====")
print(query)
print("===== Answer =====")
print(answer['result'])

===== Query =====
Explain Amazon Aurora architecture and its advantages.
===== Answer =====
Amazon Aurora architecture consists of several key components that work together to provide a highly available and scalable relational database service. Here’s an overview of its architecture and advantages:

### Architecture Components:

1. **Cluster**: An Aurora cluster consists of a primary database instance (Writer Instance) and up to 15 read replicas (Reader Instances). The primary instance handles all read and write operations, while the replicas handle only read operations.

2. **Storage Layer**: Aurora uses a log-structured distributed storage system that separates compute from storage. This allows for better scalability and performance. The storage layer is designed to automatically replicate data across multiple Availability Zones (AZs).

3. **Data Replication**: Every piece of data written to Aurora is automatically replicated six times across three Availability Zones. This ensures hi

In [8]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
# Step 5: Build RetrievalQA pipeline
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    chain_type="stuff"
)

# Step 6: Query the system
query = "Explain Amazon Aurora architecture and its advantages."
answer = qa.invoke(query)


print("===== Query =====")
print(query)
print("===== Answer =====")
print(answer['result'])

===== Query =====
Explain Amazon Aurora architecture and its advantages.
===== Answer =====
Amazon Aurora architecture consists of two main components: the Writer Instance (Primary) and Reader Instances (Read Replicas), along with a shared storage volume that spans multiple Availability Zones (AZs). 

### Architecture Components:
1. **Writer Instance (Primary)**: 
   - This instance handles all read and write operations. There is always one primary instance per cluster.

2. **Reader Instances (Read Replicas)**: 
   - These instances handle only read operations and can be up to 15 replicas of the primary instance. They help reduce the read workload on the primary database.

3. **Failover Target**: 
   - In case the primary instance fails, Aurora automatically promotes one of the reader instances to be the new primary, typically within 30 seconds.

4. **Shared Storage Volume**: 
   - Aurora uses a log-structured distributed storage system that is independent of the compute instances. Thi

In [9]:
import pandas as pd
import re

query = "What is Amazon Aurora?"

# Run similarity search with scores
faiss_results = db.similarity_search_with_score(query, k=3)
chroma_results = vectordb.similarity_search_with_score(query, k=3)

# Normalize text for readability
faiss_clean = [(re.sub(r'\n+', ' ', doc.page_content), score) for doc, score in faiss_results]
chroma_clean = [(re.sub(r'\n+', ' ', doc.page_content), score) for doc, score in chroma_results]

# Build comparison table
rows = []
for i in range(max(len(faiss_clean), len(chroma_clean))):
    faiss_text, faiss_score = faiss_clean[i] if i < len(faiss_clean) else ("-", "-")
    chroma_text, chroma_score = chroma_clean[i] if i < len(chroma_clean) else ("-", "-")
    rows.append({
        "Rank": i+1,
        "FAISS Text": faiss_text[:200] + "...",  # truncate for readability
        "FAISS Score": faiss_score,
        "Chroma Text": chroma_text[:200] + "...",
        "Chroma Score": chroma_score
    })

print(rows);

df = pd.DataFrame(rows)
print(df.to_markdown())

[{'Rank': 1, 'FAISS Text': 'Amazon Aurora - GeeksforGeeksCoursesTutorialsPracticeJobsDevOpsCloud ComputingGitAWSDockerKubernetesMicrosoft AzureGoogle Cloud PlatformPythonGolangOperating SystemComputer NetworkShare Your Experienc...', 'FAISS Score': 0.6194315, 'Chroma Text': 'Amazon Aurora - GeeksforGeeksCoursesTutorialsPracticeJobsDevOpsCloud ComputingGitAWSDockerKubernetesMicrosoft AzureGoogle Cloud PlatformPythonGolangOperating SystemComputer NetworkShare Your Experienc...', 'Chroma Score': 0.6194748878479004}, {'Rank': 2, 'FAISS Text': '-...', 'FAISS Score': '-', 'Chroma Text': 'Amazon Aurora - GeeksforGeeksCoursesTutorialsPracticeJobsDevOpsCloud ComputingGitAWSDockerKubernetesMicrosoft AzureGoogle Cloud PlatformPythonGolangOperating SystemComputer NetworkShare Your Experienc...', 'Chroma Score': 0.6194748878479004}, {'Rank': 3, 'FAISS Text': '-...', 'FAISS Score': '-', 'Chroma Text': 'Amazon Aurora - GeeksforGeeksCoursesTutorialsPracticeJobsDevOpsCloud ComputingGitAWSDockerKubernet